In [ ]:
#####主要在调试编码器和解码器

##### basic model 1125
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )
        
        # 解码器：使用一维卷积转置
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """
        将量化后的张量转换为二进制张量。
        输入: (batch_size,) 一维张量
        输出: (batch_size, bitwidth) 二维张量
        """
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            # 从最低位到最高位提取
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """
        将二进制张量转换回量化后的张量。
        输入: (batch_size, bitwidth) 二维张量
        输出: (batch_size,) 一维张量
        """
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """
        将量化值解量化为原始范围的值。
        输入: (batch_size,) 一维张量
        输出: (batch_size,) 一维张量
        """
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        
        # 编码：(batch_size, 1, 8) -> (batch_size, 1, 1)
        x_encoded = self.encoder(x)  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 确保量化前的张量是一维的 (batch_size,)
        # .view(-1) 会将任何形状（如 (batch_size, 1, 1) 或 (batch_size, 1)）安全地展平为一维
        x_encoded_1d = x_encoded.view(-1)
        
        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        
        # 解码：(batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    #optimizer = optim.SGD(codec.parameters(), lr=0.005, momentum=0.85,nesterov=True)
    optimizer = optim.AdamW(codec.parameters(),lr=1e-3,weight_decay=1e-4 )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50,T_mult=2,eta_min=0.00005) #50/30
    
    # 训练循环
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")





In [ ]:
##### transformer 0.28 or so（改动原模型，比如损失）
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import os
import math

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # 创建位置编码矩阵 (1, max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)].to(x.device)
        return self.dropout(x)

# 改进的损失函数 - 组合损失
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.alpha = alpha
        self.l1_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()
        
    def forward(self, pred, target):
        l1 = self.l1_loss(pred, target)
        mse = self.mse_loss(pred, target)
        return self.alpha * l1 + (1 - self.alpha) * mse

# Transformer 编解码器
class CodecSystem(nn.Module):
    
    def __init__(self, d_model=48, nhead=6, num_encoder_layers=3, num_decoder_layers=3, d_ff=128, dropout=0.2):
        super(CodecSystem, self).__init__()

        # 量化配置 - 调整量化范围
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -1.0  # 缩小量化范围
        self.quant_max = 1.0   # 缩小量化范围
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        # 编码器 - 增加容量
        self.encoder_proj = nn.Linear(1, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_ff, 
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu'  # 使用GELU激活函数
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        
        # 改进编码器输出
        self.encoder_final_proj = nn.Sequential(
            nn.Linear(d_model * 8, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, 1)
        )

        # 解码器
        self.quant_proj = nn.Sequential(
            nn.Linear(1, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, d_model)
        )
        self.learned_query = nn.Embedding(8, d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_ff, 
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        
        # 改进解码器输出
        self.decoder_final_proj = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))
        
        # 初始化参数
        self._init_weights()
    
    def _init_weights(self):
        """更好的权重初始化"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0, std=0.02)

    def generate_causal_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1).bool()
        return mask

    def forward(self, x):
        # x shape: (batch_size, 1, 8)
        
        # --- 编码 ---
        batch_size = x.shape[0]
        x_enc = x.transpose(1, 2)  # (batch_size, 8, 1)
        
        x_enc_proj = self.encoder_proj(x_enc)  # (batch_size, 8, d_model)
        x_enc = self.pos_enc(x_enc_proj)       # (batch_size, 8, d_model)
        
        memory = self.transformer_encoder(x_enc)  # (batch_size, 8, d_model)
        
        x_flat = memory.reshape(batch_size, -1)  # (batch_size, 8 * d_model)
        x_encoded = self.encoder_final_proj(x_flat)  # (batch_size, 1)
        x_encoded = x_encoded * self.output_scale + self.output_bias

        # --- 量化与二进制 ---
        x_quantized_1d = self.quantizer(x_encoded)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)

        # --- 解量化 ---
        x_dequantized = self.dequantize(x_quantized_recovered_1d)  # (batch_size, 1)

        # --- 解码 ---
        # 解码器只使用解量化后的信息
        
        # 1. 将解量化信息投影到特征空间
        dequantized_expanded = x_dequantized.unsqueeze(1)  # (batch_size, 1, 1)
        quant_context = self.quant_proj(dequantized_expanded)  # (batch_size, 1, d_model)
        
        # 2. 使用可学习的查询向量作为解码器输入
        learned_queries = self.learned_query.weight.unsqueeze(0)  # (1, 8, d_model)
        learned_queries = learned_queries.repeat(batch_size, 1, 1)  # (batch_size, 8, d_model)
        
        # 3. 确保quant_context的形状正确
        if len(quant_context.shape) == 2:
            quant_context = quant_context.unsqueeze(1)  # 确保是3D张量
        
        # 4. 将量化信息扩展到与查询向量相同的形状，然后相加
        quant_context_expanded = quant_context.expand(-1, 8, -1)  # (batch_size, 8, d_model)
        
        x_dec_input = learned_queries + quant_context_expanded  # (batch_size, 8, d_model)
        
        # 5. 添加位置编码
        x_dec = self.pos_enc(x_dec_input)  # (batch_size, 8, d_model)
        
        # 6. 将量化信息作为memory传递给解码器
        quant_memory = quant_context_expanded  # (batch_size, 8, d_model)
        
        causal_mask = self.generate_causal_mask(8, x.device)
        
        # 7. Transformer解码 - 查询量化memory
        x_decoded_transformer = self.transformer_decoder(
            tgt=x_dec, 
            memory=quant_memory,  # 使用量化信息作为memory
            tgt_mask=causal_mask
        )  # (batch_size, 8, d_model)

        x_decoded = self.decoder_final_proj(x_decoded_transformer)  # (batch_size, 8, 1)
        x_decoded = x_decoded.transpose(1, 2)  # (batch_size, 1, 8)
        
        return x_decoded, x_quantized_1d, x_binary

    # 量化
    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

# 数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"数据文件未找到: {data_path}")

    print(f"正在从 {data_path} 加载数据...")
    data = np.load(data_path)
    dataset = AudioDataset(data)

    # 划分数据集
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
    print(f"数据集划分完成: 训练集 {len(train_dataset)} 个样本, 验证集 {len(val_dataset)} 个样本, 测试集 {len(test_dataset)} 个样本。")

    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 初始化模型 - 使用更大的模型
    codec = CodecSystem(
        d_model=48,           # 增加特征维度
        nhead=6,              # 增加注意力头数
        num_encoder_layers=3, # 增加层数
        num_decoder_layers=3,
        d_ff=128,             # 增加前馈网络维度
        dropout=0.2           # 减少dropout
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"\n使用设备: {device}")

    # 定义损失函数和优化器
    criterion = nn.L1Loss()  # 使用组合损失

    # 改进优化器
    optimizer = optim.AdamW(
        codec.parameters(),
        lr=5e-3,           # 提高学习率
        weight_decay=1e-4,  # 调整权重衰减
        betas=(0.9, 0.98)   # 调整beta参数
    )
    
    # 改进学习率调度 - 使用OneCycleLR
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-3,
        epochs=100,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        div_factor=10.0,
        final_div_factor=100.0
    )

    # 梯度累积
    accumulation_steps = 4
    
    # 调整早停机制的 patience
    patience = 12  # 增加耐心值
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None

    # 训练循环
    num_epochs = 100
    train_loss_history = []
    val_loss_history = []

    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_train_loss = 0.0
        codec.train()
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss = loss / accumulation_steps  # 梯度累积
            loss.backward()
            
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=0.5)  # 调整梯度裁剪
                optimizer.step()
                scheduler.step()  # 每个step更新学习率
                optimizer.zero_grad()
            
            running_train_loss += loss.item() * accumulation_steps
            
            if (i + 1) % 50 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], '
                      f'Train Loss: {loss.item() * accumulation_steps:.6f}, LR: {current_lr:.2e}')

        # 处理剩余的梯度
        if len(train_loader) % accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=0.5)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)

        # 验证阶段
        running_val_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)

        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)

        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.2e}')

        # 早停逻辑
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict().copy()
            print(f'Validation loss improved! Best model state saved in memory.')
        else:
            counter += 1
            print(f'Validation loss did not improve for {counter} epochs. Patience: {counter}/{patience}')
            if counter >= patience:
                print(f"早停触发！在第 {epoch+1} 个epoch停止训练。")
                break

    plot_loss_curve(train_loss_history, val_loss_history)

    # 最终测试
    print("\n开始最终测试...")
    if best_model_state is not None:
        codec.load_state_dict(best_model_state)
    codec.eval()
    total_test_loss = 0.0
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_test_loss += loss.item() * batch_data.size(0)

    average_test_loss = total_test_loss / len(test_dataset)
    print(f"测试集平均Loss: {average_test_loss:.6f}")







In [ ]:
######Transformer + CNN 0.28
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  
import math

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # 创建位置编码矩阵 (1, max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)].to(x.device)
        return self.dropout(x)

# Transformer编码器 + CNN解码器
class CodecSystem(nn.Module):
    def __init__(self, d_model=64, nhead=8, num_encoder_layers=3, d_ff=256, dropout=0.1):
        super(CodecSystem, self).__init__()
        
        # Transformer编码器配置
        self.d_model = d_model
        self.encoder_proj = nn.Linear(1, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_ff, 
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        
        # 编码器输出投影
        self.encoder_final_proj = nn.Sequential(
            nn.Linear(d_model * 8, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, 1)
        )

        # CNN解码器 - 简化版本
        self.decoder_cnn = nn.Sequential(
            # 初始投影层，将1维输入扩展到适合CNN的维度
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.GELU(),
            
            # 上采样到2
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Upsample(scale_factor=2),  # 1 -> 2
            
            # 上采样到4
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Upsample(scale_factor=2),  # 2 -> 4
            
            # 上采样到8
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Upsample(scale_factor=2),  # 4 -> 8
            
            # 最终输出层
            nn.Conv1d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.GELU(),
            nn.Conv1d(16, 1, kernel_size=3, padding=1),
            nn.Tanh()
        )
        
        # 量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2.0  # 调整量化范围
        self.quant_max = 2.0
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        """权重初始化"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.Conv1d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def quantized_to_binary(self, quantized):
        """将量化后的张量转换为二进制张量"""
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """将二进制张量转换回量化后的张量"""
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """将量化值解量化为原始范围的值"""
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        batch_size = x.shape[0]
        
        # --- Transformer编码 ---
        # 调整输入形状: (batch_size, 1, 8) -> (batch_size, 8, 1)
        x_enc = x.transpose(1, 2)
        
        # 投影到高维空间并添加位置编码
        x_enc_proj = self.encoder_proj(x_enc)  # (batch_size, 8, d_model)
        x_enc = self.pos_enc(x_enc_proj)
        
        # Transformer编码
        memory = self.transformer_encoder(x_enc)  # (batch_size, 8, d_model)
        
        # 展平并投影到1维
        x_flat = memory.reshape(batch_size, -1)  # (batch_size, 8 * d_model)
        x_encoded = self.encoder_final_proj(x_flat)  # (batch_size, 1)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # --- 量化 ---
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # --- CNN解码 ---
        # 将解量化后的值直接作为CNN输入
        # 形状: (batch_size,) -> (batch_size, 1, 1)
        cnn_input = x_dequantized_1d.view(batch_size, 1, 1)  # (batch_size, 1, 1)
        
        # CNN上采样重建: (batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder_cnn(cnn_input)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    # 加载数据
    data = np.load(data_path)
    dataset = AudioDataset(data)
    
    # 划分数据集：训练集80%，验证集10%，测试集10%
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size]
    )
    
    print(f"数据集划分完成: 训练集 {len(train_dataset)} 个样本, 验证集 {len(val_dataset)} 个样本, 测试集 {len(test_dataset)} 个样本")
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem(
        d_model=64, 
        nhead=8, 
        num_encoder_layers=3, 
        d_ff=256, 
        dropout=0.1
    )
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # 学习率调度器
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=5, 
        verbose=True,
        min_lr=1e-6
    )
    
    # 早停机制
    patience = 10
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None
    
    # 训练循环
    num_epochs = 100
    train_loss_history = []
    val_loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        codec.train()
        running_train_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_train_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)
        
        # 验证阶段
        codec.eval()
        running_val_loss = 0.0
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)
        
        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)
        
        # 更新学习率
        scheduler.step(epoch_val_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')
        
        # 早停逻辑
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict().copy()
            print(f'验证损失改善! 保存最佳模型')
        else:
            counter += 1
            print(f'验证损失未改善 {counter}/{patience} 个epoch')
            if counter >= patience:
                print(f"早停触发! 在第 {epoch+1} 个epoch停止训练")
                break
    
    # 绘制损失曲线
    plot_loss_curve(train_loss_history, val_loss_history)
    
    # 加载最佳模型进行测试
    if best_model_state is not None:
        codec.load_state_dict(best_model_state)
        print("加载最佳模型进行测试")
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_test_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_test_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_test_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")
    
    # 保存模型
    torch.save({
        'model_state_dict': codec.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss_history': train_loss_history,
        'val_loss_history': val_loss_history,
        'best_val_loss': best_val_loss
    }, 'best_codec_model.pth')
    print("模型已保存为 'best_codec_model.pth'")





In [ ]:

#####最优超参数 1127
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积编解码器
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        # 量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    data = np.load(data_path)
    dataset = AudioDataset(data)

    # 划分训练集、验证集、测试集 (8:1:1)
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(seed)
    )

    # -------------------------- 固定训练参数（原超参数组合中选一组）--------------------------
    lr = 1e-3               # 学习率
    batch_size = 256        # 批次大小
    weight_decay = 5e-4     # 权重衰减（正则化）
    T_0 = 30                # CosineAnnealingWarmRestarts 初始周期
    T_mult = 2              # 周期倍增系数
    eta_min = 5e-5          # 最小学习率
    num_epochs = 100        # 最大训练轮数
    patience = 15           # 早停耐心值（连续15轮验证损失无改善则停止）
    # ----------------------------------------------------------------------------------------

    # 创建DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 设备配置
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    # print(f"\n训练参数:")
    # print(f"  - 学习率: {lr:.6f}")
    # print(f"  - 批次大小: {batch_size}")
    # print(f"  - 权重衰减: {weight_decay:.6f}")
    # print(f"  - 最大训练轮数: {num_epochs}")
    # print(f"  - 早停耐心值: {patience}")

    # 创建模型
    codec = CodecSystem()
    codec.to(device)

    # 损失函数、优化器、学习率调度器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(
        codec.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=T_0,
        T_mult=T_mult,
        eta_min=eta_min
    )

    # 早停相关变量
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None  # 内存保存最佳模型状态
    train_loss_history = []
    val_loss_history = []

    # 训练循环
    print("\n开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        running_train_loss = 0.0
        codec.train()
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            optimizer.step()
            running_train_loss += loss.item()

            # 每100个batch打印一次训练损失
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Train Loss: {loss.item():.6f}')

        # 计算训练集平均损失
        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)

        # 验证阶段（无梯度）
        running_val_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)  # 按批次大小加权

        # 计算验证集平均损失
        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)

        # 学习率调度器更新
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        # 打印当前轮次信息
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')

        # 早停逻辑
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict()  # 更新最佳模型状态
            print(f'  -> 验证损失改善! 最佳验证损失: {best_val_loss:.6f}')
        else:
            counter += 1
            print(f'  -> 验证损失未改善，已持续 {counter} 轮')
            if counter >= patience:
                print(f"  -> 早停触发！在第 {epoch+1} 轮停止训练")
                break

    # 绘制训练/验证损失曲线
    plot_loss_curve(train_loss_history, val_loss_history)

    # 测试阶段（使用最佳模型）
    print("\n开始测试最佳模型...")
    if best_model_state is not None:
        # 加载内存中的最佳模型状态
        codec.load_state_dict(best_model_state)
        print("已加载最佳模型状态")

        # 计算测试集损失
        total_test_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in test_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                total_test_loss += loss.item() * batch_data.size(0)

        average_test_loss = total_test_loss / len(test_dataset)
        print(f"\n测试集平均L1 Loss: {average_test_loss:.6f}")
        print(f"最佳验证集L1 Loss: {best_val_loss:.6f}")
    else:
        print("警告：未找到最佳模型状态，测试使用最后一轮模型")











In [ ]:

######transformer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import math

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 定义量化函数（保持不变）
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

# Transformer编解码器
class TransformerCodecSystem(nn.Module):
    def __init__(self, input_dim=1, d_model=64, nhead=8, num_layers=6, dim_feedforward=256, dropout=0.1):
        super(TransformerCodecSystem, self).__init__()
        
        self.d_model = d_model
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        # 编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 瓶颈层 - 降维到1维用于量化
        self.bottleneck = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
            nn.Tanh()
        )
        
        # 解码器输入投影
        self.decoder_input_proj = nn.Linear(1, d_model)
        self.pos_decoder = PositionalEncoding(d_model)
        
        # 解码器（也使用TransformerEncoder结构）
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        # 输出层
        self.output_layer = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, input_dim),
            nn.Tanh()
        )
        
        # 量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # x shape: [batch_size, 1, seq_len] -> [batch_size, seq_len, 1]
        x = x.transpose(1, 2)
        batch_size, seq_len, _ = x.shape
        
        # 编码器
        x_proj = self.input_proj(x) * math.sqrt(self.d_model)
        x_proj = self.pos_encoder(x_proj)
        x_encoded = self.encoder(x_proj)
        
        # 瓶颈层
        x_bottleneck = self.bottleneck(x_encoded)
        x_bottleneck = x_bottleneck * self.output_scale + self.output_bias
        
        # 量化
        x_bottleneck_1d = x_bottleneck.reshape(-1)
        x_quantized_1d = self.quantizer(x_bottleneck_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        
        # 反量化
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # 解码器
        x_dequantized = x_dequantized_1d.reshape(batch_size, seq_len, 1)
        x_decoder_input = self.decoder_input_proj(x_dequantized) * math.sqrt(self.d_model)
        x_decoder_input = self.pos_decoder(x_decoder_input)
        x_decoded = self.decoder(x_decoder_input)
        
        # 输出层
        output = self.output_layer(x_decoded)
        
        # 恢复原始形状 [batch_size, seq_len, 1] -> [batch_size, 1, seq_len]
        output = output.transpose(1, 2)
        
        return output, x_quantized_1d, x_binary

# 数据集类（保持不变）
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线（保持不变）
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数（基本保持不变，只修改模型创建部分）
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    data = np.load(data_path)
    dataset = AudioDataset(data)

    # 划分训练集、验证集、测试集 (8:1:1)
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(seed)
    )

    # 训练参数（可以适当调整）
    lr = 1e-4
    batch_size = 256
    weight_decay = 5e-3
    T_0 = 30
    T_mult = 2
    eta_min = 5e-5
    num_epochs = 100
    patience = 15

    # 创建DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 设备配置
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 创建Transformer模型
    codec = TransformerCodecSystem(
        input_dim=1,
        d_model=128,           # 可以调整
        nhead=8,              # 可以调整
        num_layers=6,         # 可以调整
        dim_feedforward=256,  # 可以调整
        dropout=0.2           # 可以调整
    )
    codec.to(device)
    
    print(f"模型参数量: {sum(p.numel() for p in codec.parameters()):,}")

    # 损失函数、优化器、学习率调度器（保持不变）
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(
        codec.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=T_0,
        T_mult=T_mult,
        eta_min=eta_min
    )

    # 早停相关变量（保持不变）
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None
    train_loss_history = []
    val_loss_history = []

    # 训练循环（保持不变）
    print("\n开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        running_train_loss = 0.0
        codec.train()
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            optimizer.step()
            running_train_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Train Loss: {loss.item():.6f}')

        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)

        # 验证阶段
        running_val_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)

        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)

        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')

        # 早停逻辑
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict()
            print(f'  -> 验证损失改善! 最佳验证损失: {best_val_loss:.6f}')
        else:
            counter += 1
            print(f'  -> 验证损失未改善，已持续 {counter} 轮')
            if counter >= patience:
                print(f"  -> 早停触发！在第 {epoch+1} 轮停止训练")
                break

    # 绘制损失曲线
    plot_loss_curve(train_loss_history, val_loss_history)

    # 测试阶段
    print("\n开始测试最佳模型...")
    if best_model_state is not None:
        codec.load_state_dict(best_model_state)
        print("已加载最佳模型状态")

        total_test_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in test_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                total_test_loss += loss.item() * batch_data.size(0)

        average_test_loss = total_test_loss / len(test_dataset)
        print(f"\n测试集平均L1 Loss: {average_test_loss:.6f}")
        print(f"最佳验证集L1 Loss: {best_val_loss:.6f}")
    else:
        print("警告：未找到最佳模型状态，测试使用最后一轮模型")






In [ ]:

######jiashenCNN 1127
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 加深的一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器：加深2层卷积（共5层），通道数从16→32→64→128→256→1
        self.encoder = nn.Sequential(
            # 第1层：(batch, 1, 8) → (batch, 16, 8)
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            
            # 第2层：(batch, 16, 8) → (batch, 32, 4)
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            
            # 第3层：(batch, 32, 4) → (batch, 64, 2)
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            
            # 第4层：(batch, 64, 2) → (batch, 128, 1)
            nn.Conv1d(64, 128, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            
            # 第5层：(batch, 128, 1) → (batch, 1, 1)
            nn.Conv1d(128, 1, kernel_size=1, stride=1, padding=0),
            nn.Tanh()
        )
        
        # 解码器：加深2层卷积转置（共5层），通道数从1→256→128→64→32→1
        self.decoder = nn.Sequential(
            # 第1层：(batch, 1, 1) → (batch, 128, 1)
            nn.ConvTranspose1d(1, 128, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            
            # 第2层：(batch, 128, 1) → (batch, 64, 2)
            nn.ConvTranspose1d(128, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            
            # 第3层：(batch, 64, 2) → (batch, 32, 4)
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            
            # 第4层：(batch, 32, 4) → (batch, 16, 8)
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            
            # 第5层：(batch, 16, 8) → (batch, 1, 8)
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        
        # 4比特量化配置（保持不变）
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """将量化后的张量转换为二进制张量（保持不变）"""
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """将二进制张量转换回量化后的张量（保持不变）"""
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """将量化值解量化为原始范围的值（保持不变）"""
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        
        # 编码：(batch_size, 1, 8) → (batch_size, 1, 1)
        x_encoded = self.encoder(x)  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 确保量化前的张量是一维的 (batch_size,)
        x_encoded_1d = x_encoded.view(-1)
        
        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        
        # 解码：(batch_size, 1, 1) → (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类（保持不变）
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数（保持不变）
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数（保持不变）
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 打印模型参数量（新增：方便查看加深后的参数量）
    total_params = sum(p.numel() for p in codec.parameters() if p.requires_grad)
    print(f"模型总参数量: {total_params:,}")
    
    # 损失函数和优化器（保持不变）
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=5e-5)
    
    # 训练循环（保持不变）
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段（保持不变）
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")





In [ ]:

######jiakuanCNN 1127
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 加宽的一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # --- 主要修改点: 加宽网络 ---
        # 编码器：保持4层卷积结构，将通道数从 16, 32, 64 增加到 32, 64, 128
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1),  # (batch, 32, 8)  <-- 加宽
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 4) <-- 加宽
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1),  # (batch, 128, 2) <-- 加宽
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )
        
        # 解码器：保持4层卷积转置结构，将通道数从 64, 32, 16 增加到 128, 64, 32
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 128, kernel_size=2, stride=1, padding=0),  # (batch, 128, 2) <-- 加宽
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 64, 4) <-- 加宽
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 8) <-- 加宽
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )
        
        # 4比特量化配置（保持不变）
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类（保持不变）
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数（保持不变）
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数（保持不变）
if __name__ == "__main__":
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 打印模型参数量（新增：方便查看加宽后的参数量）
    total_params = sum(p.numel() for p in codec.parameters() if p.requires_grad)
    print(f"模型总参数量: {total_params:,}")
    
    # 损失函数和优化器（保持不变）
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=5e-5)
    
    # 训练循环（保持不变）
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段（保持不变）
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")









In [ ]:


######buhao1127
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# --- 新增：一维DenseBlock和TransitionLayer ---
class DenseBlock1d(nn.Module):
    """一维密集块：每个层接收前面所有层的输出拼接作为输入"""
    def __init__(self, in_channels, growth_rate, num_layers, kernel_size=3):
        super(DenseBlock1d, self).__init__()
        self.layers = nn.ModuleList()
        
        for i in range(num_layers):
            # 每层的输入通道数 = 初始输入通道数 + 前面所有层的输出通道数（growth_rate * i）
            layer_in_channels = in_channels + growth_rate * i
            self.layers.append(nn.Sequential(
                nn.BatchNorm1d(layer_in_channels),
                nn.ReLU(inplace=True),
                nn.Conv1d(layer_in_channels, growth_rate, 
                         kernel_size=kernel_size, stride=1, padding=kernel_size//2),
            ))

    def forward(self, x):
        # 存储所有层的输出，用于拼接
        features = [x]
        for layer in self.layers:
            new_features = layer(torch.cat(features, dim=1))  # 拼接前面所有层的输出
            features.append(new_features)
        # 返回所有层输出的拼接（最终通道数 = in_channels + growth_rate * num_layers）
        return torch.cat(features, dim=1)

class TransitionLayer1d(nn.Module):
    """一维过渡层：用于压缩特征图数量和尺寸"""
    def __init__(self, in_channels, out_channels, stride=2):
        super(TransitionLayer1d, self).__init__()
        self.trans = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=1, padding=0),  # 1x1卷积压缩通道
            nn.AvgPool1d(kernel_size=2, stride=stride, padding=0)  # 平均池化压缩尺寸
        )

    def forward(self, x):
        return self.trans(x)

# --- 修改：使用DenseNet重构CodecSystem ---
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # -------------------------- 编码器（DenseNet）--------------------------
        # 初始卷积：将输入通道从1扩展到32
        self.encoder_init = nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1)
        
        # DenseBlock1 + Transition1：输入32 -> 32+16*4=96 -> 压缩到64，尺寸8->4
        self.dense1 = DenseBlock1d(in_channels=32, growth_rate=16, num_layers=4)
        self.trans1 = TransitionLayer1d(in_channels=32+16*4, out_channels=64, stride=2)
        
        # DenseBlock2 + Transition2：输入64 -> 64+16*4=128 -> 压缩到32，尺寸4->2
        self.dense2 = DenseBlock1d(in_channels=64, growth_rate=16, num_layers=4)
        self.trans2 = TransitionLayer1d(in_channels=64+16*4, out_channels=32, stride=2)
        
        # 最终编码卷积：将通道压缩到1，尺寸2->1
        self.encoder_final = nn.Conv1d(32, 1, kernel_size=2, stride=1, padding=0)
        
        # 编码激活
        self.encoder_activation = nn.Tanh()
        
        # -------------------------- 解码器（DenseNet逆过程）--------------------------
        # 初始反卷积：将1通道扩展到32，尺寸1->2
        self.decoder_init = nn.ConvTranspose1d(1, 32, kernel_size=2, stride=1, padding=0)
        
        # 逆Transition1（反卷积）：32->64，尺寸2->4
        self.decoder_trans1 = nn.ConvTranspose1d(32, 64, kernel_size=2, stride=2, padding=0, output_padding=0)
        
        # 逆DenseBlock1：64->64+16*4=128
        self.decoder_dense1 = DenseBlock1d(in_channels=64, growth_rate=16, num_layers=4)
        
        # 逆Transition2（反卷积）：128->32，尺寸4->8
        self.decoder_trans2 = nn.ConvTranspose1d(64+16*4, 32, kernel_size=2, stride=2, padding=0, output_padding=0)
        
        # 逆DenseBlock2：32->32+16*4=96
        self.decoder_dense2 = DenseBlock1d(in_channels=32, growth_rate=16, num_layers=4)
        
        # 最终反卷积：将通道压缩到1，尺寸8->8
        self.decoder_final = nn.ConvTranspose1d(32+16*4, 1, kernel_size=3, stride=1, padding=1)
        
        # 解码激活
        self.decoder_activation = nn.Tanh()
        
        # -------------------------- 量化相关（保持不变）--------------------------
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # -------------------------- 编码过程 --------------------------
        # 输入：(batch, 1, 8)
        x_enc = self.encoder_init(x)  # (batch, 32, 8)
        x_enc = self.dense1(x_enc)    # (batch, 32+16*4=96, 8)
        x_enc = self.trans1(x_enc)    # (batch, 64, 4)
        x_enc = self.dense2(x_enc)    # (batch, 64+16*4=128, 4)
        x_enc = self.trans2(x_enc)    # (batch, 32, 2)
        x_enc = self.encoder_final(x_enc)  # (batch, 1, 1)
        x_enc = self.encoder_activation(x_enc)  # (batch, 1, 1)
        
        # 量化前处理（保持不变）
        x_encoded = x_enc * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)  # (batch,)
        
        # -------------------------- 量化过程（保持不变）--------------------------
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # -------------------------- 解量化过程（保持不变）--------------------------
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)  # (batch, 1, 1)
        
        # -------------------------- 解码过程 --------------------------
        x_dec = self.decoder_init(x_dequantized_3d)  # (batch, 32, 2)
        x_dec = self.decoder_trans1(x_dec)           # (batch, 64, 4)
        x_dec = self.decoder_dense1(x_dec)           # (batch, 64+16*4=128, 4)
        x_dec = self.decoder_trans2(x_dec)           # (batch, 32, 8)
        x_dec = self.decoder_dense2(x_dec)           # (batch, 32+16*4=96, 8)
        x_dec = self.decoder_final(x_dec)            # (batch, 1, 8)
        x_decoded = self.decoder_activation(x_dec)   # (batch, 1, 8)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类（保持不变）
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数（保持不变）
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数（保持不变）
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 打印模型参数量（新增：查看DenseNet模型大小）
    total_params = sum(p.numel() for p in codec.parameters() if p.requires_grad)
    print(f"模型总参数量: {total_params:,}")
    
    # 损失函数和优化器（保持不变）
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2, eta_min=0.00005)
    
    # 训练循环（保持不变）
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss with DenseNet)")
    
    # 测试阶段（保持不变）
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")

